# E2: horizon sweep across the 4,096-token window
Run on HF Jobs L4 (vLLM + prefix caching). Then fit step-vs-smooth per curve.

In [ ]:
!pip uninstall -y torchao
!git clone -q -b feat/hsafe https://github.com/dhairya-pandya/Lexsi-AI-Alignment repo
%cd repo
!pip install -q -e '.[gpu]'

In [ ]:
import os
# Kaggle: from kaggle_secrets import UserSecretsClient; os.environ['HF_TOKEN']=UserSecretsClient().get_secret('HF_TOKEN')
# HF Jobs: pass --secrets HF_TOKEN

In [ ]:
for c in ['e2_global_f2','e2_global_f1','e2_global_f3','e2_expanse_f2']:
    !python -m hsafe.run --config configs/{c}.yaml

In [ ]:
import pandas as pd
from hsafe.stats import step_vs_smooth
rows = []
for run in ['e2_global_f1','e2_global_f2','e2_global_f3','e2_expanse_f2']:
    items = pd.read_csv(f'results/{run}/items.csv')
    for (cond, kind, metric), g in items.groupby(['condition','kind','metric']):
        if metric == 'p_refusal_first':
            continue
        res = step_vs_smooth(g['d'].to_numpy(), g['value'].to_numpy())
        rows.append(dict(run=run, condition=cond, kind=kind, metric=metric, **{k: v for k, v in res.items() if k != 'jump_ci'}, jump_lo=res['jump_ci'][0], jump_hi=res['jump_ci'][1]))
import os; os.makedirs('results/e2', exist_ok=True)
pd.DataFrame(rows).to_csv('results/e2/step_tests.csv', index=False)
pd.DataFrame(rows)